# GPT-2 Mythology Fine-Tuning (RESUME from 2hr checkpoint)
**Resume fine-tuning GPT-2 (124M) on Indian Mythology + World Folklore**

Datasets: `rahular/itihasa` (Ramayana/Mahabharata) + `merve/folk-mythology-tales` + `KyiThinNu/FairyTales`

Resuming from existing 2hr checkpoint on HuggingFace Hub | ~3 hours remaining on T4

In [ ]:
!pip install -q torch transformers datasets huggingface_hub python-dotenv accelerate

In [ ]:
import os, time, math
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from huggingface_hub import HfApi, login, hf_hub_download

# ============ CONFIGURATION ============
HF_TOKEN = os.environ.get('hf', 'YOUR_HF_TOKEN')  # Set in Kaggle secrets or .env
HF_REPO  = 'blaze1234567/gpt2'
OUTPUT_DIR = './gpt2-mythology'

# Training hyperparameters (tuned for ~3 hrs remaining on T4)
BATCH_SIZE = 8
GRAD_ACCUM = 4          # effective batch = 32
EPOCHS = 2              # 2 more epochs on top of existing checkpoint
LR = 3e-5              # slightly lower LR for continued training
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100      # short warmup since we are resuming
MAX_LENGTH = 512
FREEZE_LAYERS = 6       # freeze bottom 6 of 12 transformer layers
FP16 = torch.cuda.is_available()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPUs: {torch.cuda.device_count()}')
login(token=HF_TOKEN)

## 1. Load & Prepare Datasets

In [ ]:
print('Loading datasets from HuggingFace...')

# Dataset 1: Itihasa — Ramayana & Mahabharata (English translations)
ds_itihasa = load_dataset('rahular/itihasa', split='train', trust_remote_code=True)
# Extract English text from translation pairs
itihasa_texts = ds_itihasa.map(
    lambda x: {'text': x['translation']['en']},
    remove_columns=ds_itihasa.column_names
)
print(f'  Itihasa: {len(itihasa_texts)} examples')

# Dataset 2: Folk Mythology Tales
ds_folk = load_dataset('merve/folk-mythology-tales', split='train', trust_remote_code=True)
# This dataset has a 'text' column already
print(f'  Folk Mythology: {len(ds_folk)} examples')

# Dataset 3: Fairy Tales (narrative structure)
try:
    ds_fairy = load_dataset('KyiThinNu/FairyTales', split='train', trust_remote_code=True)
    # Normalize column name
    if 'text' not in ds_fairy.column_names:
        col = ds_fairy.column_names[0]
        ds_fairy = ds_fairy.rename_column(col, 'text')
    ds_fairy = ds_fairy.select_columns(['text'])
    print(f'  FairyTales: {len(ds_fairy)} examples')
except Exception as e:
    print(f'  FairyTales skipped: {e}')
    ds_fairy = None

# Combine all datasets
all_ds = [itihasa_texts, ds_folk.select_columns(['text'])]
if ds_fairy is not None:
    all_ds.append(ds_fairy)

combined = concatenate_datasets(all_ds)

# Filter out empty/short texts
combined = combined.filter(lambda x: x['text'] is not None and len(x['text'].strip()) > 50)
print(f'\nTotal combined dataset: {len(combined)} examples')
print(f'Sample: "{combined[0]["text"][:120]}..."')

## 2. Tokenization

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )

print('Tokenizing dataset...')
tokenized = combined.map(tokenize_fn, batched=True, remove_columns=['text'], num_proc=2)

# Train/val split (95/5)
split = tokenized.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
val_ds = split['test']
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

# Estimate tokens
avg_tokens = sum(sum(1 for t in x['input_ids'] if t != tokenizer.pad_token_id) for x in train_ds.select(range(min(1000, len(train_ds))))) / min(1000, len(train_ds))
total_tokens = avg_tokens * len(train_ds)
print(f'Estimated total tokens: {total_tokens/1e6:.1f}M')

## 3. Load Model from Existing 2hr Checkpoint & Freeze Layers

In [ ]:
# Load the EXISTING 2-hour checkpoint instead of base GPT-2
print('Loading existing 2hr checkpoint from HuggingFace Hub...')
try:
    ckpt_path = hf_hub_download(
        repo_id=HF_REPO,
        filename='gpt2-mythology/mythology_checkpoint.pt',
        token=HF_TOKEN,
    )
    # Start with base GPT-2 architecture, then load trained weights
    model = GPT2LMHeadModel.from_pretrained('gpt2')
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state_dict = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state_dict, strict=False)
    prev_epoch = ckpt.get('epoch', '?') if isinstance(ckpt, dict) else '?'
    prev_step = ckpt.get('global_step', '?') if isinstance(ckpt, dict) else '?'
    prev_loss = ckpt.get('best_val_loss', '?') if isinstance(ckpt, dict) else '?'
    print(f'  Loaded checkpoint: epoch={prev_epoch}, step={prev_step}, val_loss={prev_loss}')
    print(f'  Resuming training from these weights!')
except Exception as e:
    print(f'  Could not load checkpoint ({e}), falling back to base GPT-2')
    model = GPT2LMHeadModel.from_pretrained('gpt2')

# Freeze bottom layers to prevent overfitting on smaller dataset
for i, layer in enumerate(model.transformer.h):
    if i < FREEZE_LAYERS:
        for param in layer.parameters():
            param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total/1e6:.1f}M')
print(f'Trainable:    {trainable/1e6:.1f}M ({100*trainable/total:.1f}%)')
print(f'Frozen:       {(total-trainable)/1e6:.1f}M (bottom {FREEZE_LAYERS} layers)')

## 4. Hourly Checkpoint Callback

In [ ]:
class HourlyHubCheckpoint(TrainerCallback):
    """Upload checkpoint to HuggingFace Hub every hour."""

    def __init__(self, repo_id, token, interval_sec=3600):
        self.repo_id = repo_id
        self.token = token
        self.interval = interval_sec
        self.last_save = time.time()
        self.api = HfApi(token=token)
        try:
            self.api.create_repo(repo_id, exist_ok=True, private=True)
        except Exception:
            pass

    def on_log(self, args, state, control, **kwargs):
        now = time.time()
        if now - self.last_save >= self.interval:
            self._upload(args, state, kwargs.get('model'))
            self.last_save = now

    def on_train_end(self, args, state, control, **kwargs):
        self._upload(args, state, kwargs.get('model'))

    def _upload(self, args, state, model):
        print(f'\nHourly upload at step {state.global_step}...')
        save_dir = os.path.join(args.output_dir, 'hourly_upload')
        if model is not None:
            model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)
        # Also save a .pt checkpoint with metadata
        if model is not None:
            torch.save({
                'model': model.state_dict(),
                'epoch': state.epoch,
                'global_step': state.global_step,
                'best_val_loss': state.best_metric,
            }, os.path.join(save_dir, 'mythology_checkpoint.pt'))
        try:
            self.api.upload_folder(
                folder_path=save_dir,
                path_in_repo='gpt2-mythology',
                repo_id=self.repo_id,
            )
            print(f'  Uploaded to {self.repo_id}/gpt2-mythology')
        except Exception as e:
            print(f'  Upload failed: {e}')

print('Hourly checkpoint callback ready.')

## 5. Training (Resumed)

In [ ]:
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    fp16=FP16,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    dataloader_num_workers=2,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[HourlyHubCheckpoint(HF_REPO, HF_TOKEN, interval_sec=3600)],
)

print(f'\nStarting training (RESUMED from 2hr checkpoint)...')
print(f'   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')
print(f'   Epochs: {EPOCHS}')
print(f'   LR: {LR} (lower for continued training)')
print(f'   Frozen layers: bottom {FREEZE_LAYERS}/12')
print(f'   FP16: {FP16}')
print(f'   Hourly checkpoints -> {HF_REPO}\n')

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 3600
print(f'\nTraining complete in {elapsed:.1f} hours!')

## 6. Final Upload to HuggingFace Hub

In [ ]:
# Save final model locally
final_dir = os.path.join(OUTPUT_DIR, 'final')
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

# Save .pt checkpoint too
torch.save({
    'model': model.state_dict(),
    'epoch': EPOCHS,
    'global_step': trainer.state.global_step,
    'best_val_loss': trainer.state.best_metric,
    'config': {
        'freeze_layers': FREEZE_LAYERS,
        'lr': LR, 'epochs': EPOCHS,
        'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM,
    }
}, os.path.join(final_dir, 'mythology_checkpoint.pt'))

# Upload to Hub
api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path=final_dir,
    path_in_repo='gpt2-mythology',
    repo_id=HF_REPO,
)
print(f'\nFinal model uploaded to {HF_REPO}/gpt2-mythology')

## 7. Test Generation (Before vs After)

In [ ]:
from transformers import pipeline

# Load raw GPT-2 for comparison
raw_gen = pipeline('text-generation', model='gpt2', device=0 if torch.cuda.is_available() else -1)
# Fine-tuned model
myth_gen = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

prompts = [
    'Once there lived a king named Dasharatha who',
    'In the great battle of Kurukshetra, Arjuna',
    'The sage sat beneath the banyan tree and',
    'Long ago, when gods walked among mortals,',
]

gen_kwargs = dict(max_new_tokens=150, do_sample=True, temperature=0.8,
                  top_k=50, top_p=0.9, repetition_penalty=1.3,
                  no_repeat_ngram_size=3)

for p in prompts:
    print('='*70)
    print(f'PROMPT: {p}')
    print('-'*70)
    raw_out = raw_gen(p, **gen_kwargs)[0]['generated_text']
    print(f'RAW GPT-2:\n{raw_out}\n')
    myth_out = myth_gen(p, **gen_kwargs)[0]['generated_text']
    print(f'MYTHOLOGY GPT-2:\n{myth_out}')
    print('='*70 + '\n')

In [ ]:
# Training metrics summary
logs = trainer.state.log_history
train_losses = [(l['step'], l['loss']) for l in logs if 'loss' in l]
eval_losses = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

print('\nTraining Summary')
print('='*40)
if train_losses:
    print(f'  Start loss:  {train_losses[0][1]:.4f}')
    print(f'  Final loss:  {train_losses[-1][1]:.4f}')
if eval_losses:
    print(f'  Best val:    {min(l[1] for l in eval_losses):.4f}')
    print(f'  Final val:   {eval_losses[-1][1]:.4f}')
print(f'  Total steps: {trainer.state.global_step}')
print(f'  Total time:  {elapsed:.1f} hours')